[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/13_simple_search/notebooks/01_grafos.ipynb)


# Notebook 1: Grafos — Fundamentos y Representaciones

En las notas leíste las definiciones formales. Ahora toca **construirlos, visualizarlos y medir su comportamiento** en Python.

Un grafo $G = (V, E)$ es una estructura matemática con un conjunto de **vértices** $V$ y un conjunto de **aristas** $E \subseteq V \times V$. En este notebook veremos:

1. Cómo representar grafos como diccionarios de Python y matrices de NumPy
2. Cómo medir propiedades básicas (grado, densidad, componentes conexas)
3. El impacto de la densidad en el rendimiento de las operaciones

> **Prerequisito:** haber leído `01_grafos.md` en las notas del curso.


In [ ]:
# Instalar dependencias (ejecutar solo en Colab)
# !pip install numpy matplotlib networkx


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import time
import sys

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "primary": "#2563EB",
    "secondary": "#10B981",
    "accent": "#F59E0B",
    "danger": "#EF4444",
    "node": "#93C5FD",
    "edge": "#475569",
}

print("Dependencias cargadas correctamente.")


---
## Sección 1: Construyendo Grafos

La representación más directa de un grafo en Python es un **diccionario de listas de adyacencia**:
cada clave es un vértice y su valor es la lista de vértices vecinos.

Para un grafo **no dirigido** con arista $\{u, v\}$, la representamos dos veces: $u \to v$ y $v \to u$.
Para un **dígrafo**, cada arista $(u, v)$ aparece solo en la lista de $u$.

### Ventajas del diccionario de adyacencia
- Iterar sobre vecinos de $u$: $O(\deg(u))$ — solo recorremos los vecinos reales
- Agregar un vértice o arista: $O(1)$ amortizado
- Espacio: $O(V + E)$ — eficiente para grafos dispersos


In [ ]:
# Grafo no dirigido: 6 nodos, estilo lista de adyacencia
grafo_nd = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E'],
}

# Construir el mismo grafo con NetworkX para visualizarlo
G_nd = nx.Graph()
for u, vecinos in grafo_nd.items():
    for v in vecinos:
        G_nd.add_edge(u, v)

fig, ax = plt.subplots(figsize=(7, 5))
pos = nx.spring_layout(G_nd, seed=7)
nx.draw_networkx(G_nd, pos=pos, ax=ax,
                 node_color=COLORS["node"], node_size=800,
                 edge_color=COLORS["edge"], width=2,
                 font_size=13, font_weight='bold')
ax.set_title("Grafo no dirigido — 6 vértices, 5 aristas", fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Vértices: {list(G_nd.nodes())}")
print(f"Aristas:  {list(G_nd.edges())}")
print(f"|V| = {G_nd.number_of_nodes()}, |E| = {G_nd.number_of_edges()}")


In [ ]:
# Dígrafo: 5 nodos
grafo_dir = {
    'S': ['A', 'B'],
    'A': ['C'],
    'B': ['C', 'D'],
    'C': ['D'],
    'D': [],
}

G_dir = nx.DiGraph()
for u, vecinos in grafo_dir.items():
    for v in vecinos:
        G_dir.add_edge(u, v)

fig, ax = plt.subplots(figsize=(7, 5))
pos = nx.spring_layout(G_dir, seed=42)
nx.draw_networkx(G_dir, pos=pos, ax=ax,
                 node_color=COLORS["accent"], node_size=800,
                 edge_color=COLORS["edge"], width=2,
                 arrows=True, arrowsize=20,
                 font_size=13, font_weight='bold')
ax.set_title("Dígrafo — 5 vértices, 6 aristas", fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Vértices: {list(G_dir.nodes())}")
print(f"Aristas:  {list(G_dir.edges())}")


---
## Sección 2: Matrices de Adyacencia

La **matriz de adyacencia** $A \in \{0,1\}^{V \times V}$ define:

$$A[i][j] = \begin{cases} 1 & \text{si } (i,j) \in E \\ 0 & \text{si no} \end{cases}$$

Para grafos no dirigidos, $A$ es **simétrica**: $A[i][j] = A[j][i]$.

### Complejidades
| Operación | Lista de adyacencia | Matriz de adyacencia |
|-----------|--------------------|-----------------------|
| ¿Existe arista $(u,v)$? | $O(\deg(u))$ | $O(1)$ |
| Vecinos de $u$ | $O(\deg(u))$ | $O(V)$ |
| Espacio | $O(V + E)$ | $O(V^2)$ |

La matriz es preferible cuando $E \approx V^2$ (grafos **densos**); la lista lo es cuando $E \ll V^2$ (grafos **dispersos**).


In [ ]:
# Convertir diccionario de adyacencia a matriz numpy
def dict_a_matriz(grafo):
    nodos = sorted(grafo.keys())
    idx = {n: i for i, n in enumerate(nodos)}
    n = len(nodos)
    A = np.zeros((n, n), dtype=int)
    for u, vecinos in grafo.items():
        for v in vecinos:
            A[idx[u], idx[v]] = 1
    return A, nodos

A, nodos = dict_a_matriz(grafo_nd)
print("Matriz de adyacencia del grafo no dirigido:")
print("Nodos:", nodos)
print(A)
print(f"\nEs simétrica: {np.array_equal(A, A.T)}")

# Visualizar la matriz
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(A, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(nodos))); ax.set_xticklabels(nodos)
ax.set_yticks(range(len(nodos))); ax.set_yticklabels(nodos)
for i in range(len(nodos)):
    for j in range(len(nodos)):
        ax.text(j, i, str(A[i,j]), ha='center', va='center', fontsize=12,
                color='white' if A[i,j] else 'black')
ax.set_title("Matriz de adyacencia $A$", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Verificar complejidades
idx = {n: i for i, n in enumerate(nodos)}

# Consulta de arista: O(1) con matriz
u, v = 'A', 'B'
existe = bool(A[idx[u], idx[v]])
print(f"¿Existe arista ({u},{v})? {existe}  ← O(1) con matriz")

# Iterar vecinos: O(V) con matriz (recorremos toda la fila)
fila_A = A[idx['A']]
vecinos_de_A_matriz = [nodos[j] for j, val in enumerate(fila_A) if val]
vecinos_de_A_lista  = grafo_nd['A']
print(f"Vecinos de A (matriz, O(V)={len(nodos)}): {vecinos_de_A_matriz}")
print(f"Vecinos de A (lista,  O(deg)={len(vecinos_de_A_lista)}): {vecinos_de_A_lista}")
print("\n→ Con la lista de adyacencia solo tocamos los vecinos reales.")
print("  Con la matriz recorremos TODOS los nodos aunque la mayoría sean 0.")


---
## Sección 3: Grado de Vértices

El **grado** $\deg(v)$ de un vértice es el número de aristas incidentes.

- En un grafo no dirigido: $\deg(v) = |\{u : \{u,v\} \in E\}|$
- En un dígrafo distinguimos **grado de entrada** $\deg^-(v)$ y **grado de salida** $\deg^+(v)$

El **lema del apretón de manos** establece:
$$\sum_{v \in V} \deg(v) = 2|E|$$

Esto se debe a que cada arista contribuye 2 al total (una por cada extremo).


In [ ]:
# Grado en grafo no dirigido
print("=== Grado en grafo no dirigido ===")
for nodo, vecinos in sorted(grafo_nd.items()):
    print(f"  deg({nodo}) = {len(vecinos)}")
total_grado = sum(len(v) for v in grafo_nd.values())
num_aristas = total_grado // 2
print(f"\nΣ deg = {total_grado} = 2 × {num_aristas} aristas  ✓")

print("\n=== Grado en y salida en dígrafo ===")
for nodo in sorted(grafo_dir.keys()):
    out_deg = len(grafo_dir[nodo])
    in_deg  = sum(1 for u in grafo_dir for v in grafo_dir[u] if v == nodo)
    print(f"  {nodo}: deg⁺ (salida) = {out_deg},  deg⁻ (entrada) = {in_deg}")


In [ ]:
# Histograma de grados — grafo aleatorio de Erdős–Rényi
np.random.seed(42)
G_er = nx.erdos_renyi_graph(n=50, p=0.15, seed=42)
grados = [d for _, d in G_er.degree()]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribución de grado
axes[0].hist(grados, bins=range(0, max(grados)+2), color=COLORS["primary"],
             edgecolor='white', alpha=0.85)
axes[0].axvline(np.mean(grados), color=COLORS["danger"], linestyle='--',
                linewidth=2, label=f'Media = {np.mean(grados):.2f}')
axes[0].set_xlabel("Grado $k$", fontsize=12)
axes[0].set_ylabel("Frecuencia", fontsize=12)
axes[0].set_title("Distribución de grado\n(Erdős–Rényi, n=50, p=0.15)", fontsize=12)
axes[0].legend()

# Visualizar el grafo coloreado por grado
pos = nx.spring_layout(G_er, seed=0)
nx.draw_networkx_edges(G_er, pos, ax=axes[1], alpha=0.4, edge_color=COLORS["edge"])
nc = nx.draw_networkx_nodes(G_er, pos, ax=axes[1], node_color=grados,
                             cmap=plt.cm.YlOrRd, node_size=200)
plt.colorbar(nc, ax=axes[1], label='Grado')
axes[1].set_title("Grafo coloreado por grado", fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"Grado mínimo: {min(grados)}, máximo: {max(grados)}, medio: {np.mean(grados):.2f}")
print(f"Verificación lema del apretón: Σdeg = {sum(grados)}, 2|E| = {2*G_er.number_of_edges()}")


---
## Sección 4: Componentes Conexas

Una **componente conexa** de un grafo no dirigido es un subconjunto maximal $C \subseteq V$ tal que
para todo par $u, v \in C$ existe un camino de $u$ a $v$.

Para encontrarlas, usamos una variante de **DFS** (Depth-First Search):
1. Marcar todos los vértices como no visitados
2. Para cada vértice no visitado, hacer DFS desde él — todo lo alcanzado forma una componente

La complejidad es $O(V + E)$ — visitamos cada vértice y cada arista exactamente una vez.


In [ ]:
# Grafo desconectado con 3 componentes
grafo_desc = {
    'A': ['B', 'C'],  # componente 1
    'B': ['A'],
    'C': ['A'],
    'D': ['E'],       # componente 2
    'E': ['D', 'F'],
    'F': ['E'],
    'G': [],          # componente 3 (nodo aislado)
}

def dfs_componente(grafo, inicio, visitados):
    """DFS iterativo que devuelve todos los nodos alcanzables desde 'inicio'."""
    componente = []
    pila = [inicio]
    while pila:
        nodo = pila.pop()
        if nodo in visitados:
            continue
        visitados.add(nodo)
        componente.append(nodo)
        for vecino in grafo[nodo]:
            if vecino not in visitados:
                pila.append(vecino)
    return componente

def componentes_conexas(grafo):
    """Devuelve una lista de componentes (cada componente es una lista de nodos)."""
    visitados = set()
    componentes = []
    for nodo in grafo:
        if nodo not in visitados:
            comp = dfs_componente(grafo, nodo, visitados)
            componentes.append(sorted(comp))
    return componentes

comps = componentes_conexas(grafo_desc)
print(f"Número de componentes conexas: {len(comps)}")
for i, c in enumerate(comps, 1):
    print(f"  Componente {i}: {c}")


In [ ]:
# Visualizar el grafo desconectado
G_desc = nx.Graph()
for u, vecinos in grafo_desc.items():
    G_desc.add_node(u)
    for v in vecinos:
        G_desc.add_edge(u, v)

comp_colors = {}
palette = [COLORS["primary"], COLORS["secondary"], COLORS["accent"]]
for i, comp in enumerate(comps):
    for nodo in comp:
        comp_colors[nodo] = palette[i]

node_colors = [comp_colors[n] for n in G_desc.nodes()]

fig, ax = plt.subplots(figsize=(8, 5))
pos = nx.spring_layout(G_desc, seed=5)
nx.draw_networkx(G_desc, pos=pos, ax=ax,
                 node_color=node_colors, node_size=900,
                 edge_color=COLORS["edge"], width=2,
                 font_size=13, font_weight='bold', font_color='white')
ax.set_title("Grafo con 3 componentes conexas (colores distintos)", fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()


---
## Sección 5: Representación vs. Densidad

La **densidad** de un grafo no dirigido se define como:

$$\rho = \frac{2|E|}{|V|(|V|-1)}$$

- $\rho \approx 0$: grafo **disperso** (sparse) — la lista de adyacencia es eficiente en memoria
- $\rho \approx 1$: grafo **denso** (dense) — la matriz de adyacencia ocupa similar espacio pero da consultas $O(1)$

En algoritmos de búsqueda (BFS, DFS), el coste total es $O(V + E)$:
- Grafo disperso con $E = O(V)$: costo $O(V)$
- Grafo denso con $E = O(V^2)$: costo $O(V^2)$

Elegir bien la representación **no cambia la complejidad asintótica** pero sí el factor constante y el uso de memoria.


In [ ]:
# Grafo disperso: 100 nodos, ~120 aristas
G_sparse = nx.gnm_random_graph(n=100, m=120, seed=1)

# Grafo denso: 20 nodos, ~380 aristas (de 190 posibles → casi completo)
G_dense = nx.gnm_random_graph(n=20, m=190, seed=2)

def densidad(G):
    n = G.number_of_nodes()
    e = G.number_of_edges()
    return 2*e / (n*(n-1)) if n > 1 else 0

print("Grafo disperso:")
print(f"  |V| = {G_sparse.number_of_nodes()}, |E| = {G_sparse.number_of_edges()}")
print(f"  Densidad ρ = {densidad(G_sparse):.4f}")

print("\nGrafo denso:")
print(f"  |V| = {G_dense.number_of_nodes()}, |E| = {G_dense.number_of_edges()}")
print(f"  Densidad ρ = {densidad(G_dense):.4f}")


In [ ]:
# Comparar memoria: lista de adyacencia vs matriz
def memoria_lista(G):
    """Bytes estimados de la lista de adyacencia."""
    total = 0
    for nodo in G.nodes():
        total += sys.getsizeof(list(G.neighbors(nodo)))
    return total

def memoria_matriz(n):
    """Bytes de una matriz numpy n×n de enteros."""
    return np.zeros((n, n), dtype=np.int8).nbytes

print("=== Uso de memoria ===")
for nombre, G in [("Disperso (100n, 120e)", G_sparse), ("Denso (20n, 190e)", G_dense)]:
    n = G.number_of_nodes()
    ml = memoria_lista(G)
    mm = memoria_matriz(n)
    print(f"\n{nombre}:")
    print(f"  Lista de adyacencia: {ml:6} bytes")
    print(f"  Matriz de adyacencia:{mm:6} bytes")
    print(f"  Razón matriz/lista:  {mm/ml:.2f}x")

print("\n→ Para el grafo disperso, la matriz ocupa MUCHO más memoria.")
print("  Para el grafo denso, la diferencia se achica.")


In [ ]:
# Comparar tiempo de consulta de arista
# Construir ambas representaciones para G_sparse
nodos_s = list(G_sparse.nodes())
idx_s = {n: i for i, n in enumerate(nodos_s)}
A_sparse = nx.to_numpy_array(G_sparse, dtype=np.int8)
adj_list_sparse = {n: set(G_sparse.neighbors(n)) for n in nodos_s}

N_consultas = 100_000
pares = [(nodos_s[i % len(nodos_s)], nodos_s[(i*7+3) % len(nodos_s)])
         for i in range(N_consultas)]

# Tiempo con matriz
t0 = time.perf_counter()
for u, v in pares:
    _ = A_sparse[idx_s[u], idx_s[v]]
t_matriz = time.perf_counter() - t0

# Tiempo con lista
t0 = time.perf_counter()
for u, v in pares:
    _ = v in adj_list_sparse[u]
t_lista = time.perf_counter() - t0

print(f"Consulta de arista ({N_consultas:,} veces) — Grafo disperso (100 nodos):")
print(f"  Matriz numpy:        {t_matriz*1000:.2f} ms")
print(f"  Lista (set lookup):  {t_lista*1000:.2f} ms")
print()
print("Nota: con sets Python, la consulta en lista también es O(1) amortizado.")
print("La diferencia real aparece al *iterar* vecinos, no al consultar aristas.")


---
## 🔧 Ejercicios

### Ejercicio 1: Grafo de palabras

Dado un vocabulario de palabras de 4 letras, construye un grafo donde dos palabras están conectadas si difieren en exactamente **una** letra (ej: `GATO` ↔ `DATO`).

1. Implementa la función `construir_grafo_palabras(vocab)` que devuelve un dict de adyacencia.
2. Encuentra las palabras con mayor grado.
3. ¿Cuántas componentes conexas tiene el grafo?

```python
vocab = ['GATO', 'DATO', 'PATO', 'RATO', 'RATA', 'DATA', 'RUTA', 'MUTA', 'MULA', 'BALA']
# Tu código aquí
```

### Ejercicio 2: Teorema de Euler

Un grafo no dirigido tiene un **camino euleriano** (que recorre cada arista exactamente una vez) si y solo si tiene exactamente 0 o 2 vértices de grado impar.

Escribe una función `tiene_camino_euleriano(grafo)` que verifique esta condición. Pruébala con:
- El grafo no dirigido de la Sección 1
- Un ciclo de 5 nodos
- El grafo de los Puentes de Königsberg (el grafo clásico que Euler estudió)

```python
# Puentes de Königsberg: 4 nodos (A=tierra norte, B=isla, C=tierra sur, D=tierra este)
konigsberg = {
    'A': ['B', 'B', 'D'],     # dos puentes a B, uno a D
    'B': ['A', 'A', 'C', 'C', 'D'],
    'C': ['B', 'B', 'D'],
    'D': ['A', 'B', 'C'],
}
# Tu código aquí
```

### Ejercicio 3: Densidad y búsqueda

Genera grafos Erdős–Rényi con $n = 50$ nodos y probabilidad $p \in \{0.05, 0.1, 0.2, 0.4, 0.6, 0.8\}$.

Para cada grafo, elige dos nodos al azar y ejecuta BFS (puedes usar `nx.bfs_edges`). Mide:
1. El número de nodos expandidos (frontera total)
2. La longitud del camino más corto

Grafica ambas métricas en función de $p$. ¿Qué patrón observas?
